# App 5 · MCP — 让一份 server 在所有 LLM 之间通用

2024 年中之前，要让一个内部 CRM 的 `query_order` 函数被 GPT-4 调到，你得给 OpenAI 写一份 `function_call` schema；要让 Claude 也能调，再写一份 `tool_use` schema；接 Cursor IDE 又得写一份 Cursor 自己的格式。同一个函数，三套描述，每次新模型出来都得重写。

Anthropic 在 2024 年 11 月放出 [MCP（Model Context Protocol）](https://modelcontextprotocol.io/) 就是为了终结这件事。它没发明新算法、新模型，只是定义了一套**工具调用的传输协议**：写一次 server，所有支持 MCP 的 LLM 客户端（Claude Desktop、Cursor、Claude Code、自家 app）都能跟它对话。

这一节我们做三件事——把 MCP 的三类原语（Tools / Resources / Prompts）讲清楚；用纯 Python 跑一遍真协议（subprocess + stdio JSON-RPC，看到完整的 initialize / tools/list / tools/call / shutdown 帧）；最后加上权限和容错。这两件事是把 demo 推到生产前必补的功课。

> **跑这一节前**：跑过 [App0](./App0_Setup_Check.ipynb) 把环境就绪，再跑过 App1–App4 理解 Agent 的工具调用语义。可选 SDK `pip install 'mcp>=0.9'` 不装也行——`utils/mcp_helpers.py` 里有一个 in-process 教学版 `EduMCPServer` 可以兜底，API 跟官方 SDK 故意保持一致，后面切换只是 `pip install` 一行的事。

In [1]:
# 自动定位 repo 根目录，让 utils 可以 import
import os, sys
_cur = os.path.abspath("")
_root = None
for _c in [_cur, os.path.dirname(_cur), os.path.dirname(os.path.dirname(_cur))]:
    if os.path.isdir(os.path.join(_c, "utils")) and os.path.isfile(os.path.join(_c, "README.md")):
        _root = _c; break
if _root is None:
    raise RuntimeError("找不到 repo 根目录")
os.chdir(_root); sys.path.insert(0, _root)
print(f"📂 repo root: {_root}")


📂 repo root: C:\Users\lvbab\Documents\GitHub\LLM-Agent-Core_Concept_Code


## 1. MCP 之前的世界长什么样

让我们先把"为什么需要 MCP"看清楚。在 2024 年中之前，把"LLM 调内部 CRM"做出来的工程师要走这套流程：

```
工程师写 Python 函数 query_order(order_id)
        ↓
为 OpenAI 写一份 JSON schema → 塞进 function_call
为 Anthropic 再写一份 JSON schema → 塞进 tool_use
为 Google Gemini 再写一份      → function_declarations
（要接 Cursor / Claude Code / Cline 各家 IDE，再各写一份）
        ↓
任何一个上游 schema 改格式，每家都要跟着改
```

这是经典的 **N×M 集成问题**：N 个工具乘以 M 个客户端。每加一个新 LLM，得把 N 个工具的 schema 重写一遍。

更糟糕的是这些 schema 不只是格式不同，**语义也不同**——OpenAI 的 `function_call` 一次只能返回一个工具调用；Anthropic 早期的 `tool_use` 支持并行；Google 把工具描述放在 `tool_config` 而不是 `tools` 字段里。要做一层 adapter 屏蔽这些差异，本身就是个工程项目。

MCP 把这件事翻面看：**不要让每个 LLM 客户端各自定义"工具长什么样"，而是让 server 端统一暴露一套接口，所有客户端去对接这一套**。这跟 USB-C 取代 micro-USB / Lightning / Type-A 是同一个故事——重要的不是"接口设计得多漂亮"，而是"统一了"。统一之后写一次能跑很多地方，整个生态就转起来。

## 2. 协议骨架：三类原语 + JSON-RPC

MCP 最关键的设计决定是把 server 暴露的东西**分成三类原语**——Tools、Resources、Prompts。这种分类看起来朴素，但它决定了 client 怎么协商、怎么做权限、怎么 cache。

形式上，MCP server 提供四个命名空间的 RPC 方法：

| 命名空间 | RPC 方法 | 干什么 |
|---|---|---|
| `tools/` | `tools/list`, `tools/call` | 列出 / 调用可执行函数（带 JSON schema） |
| `resources/` | `resources/list`, `resources/read` | 列出 / 读取数据源（文件、DB、API 端点） |
| `prompts/` | `prompts/list`, `prompts/get` | 列出 / 渲染可复用的提示模板 |
| 元方法 | `initialize`, `shutdown` | 握手 / 优雅退出 |

为什么要分三类，不是统一成"工具"一种？因为它们的**调用语义和权限粒度不同**：

- **Tool 是动词**——参数是输入，调用有副作用（写数据库、发通知）。需要细粒度的执行权限和审计日志。
- **Resource 是名词**——用 URI 标识（`file:///docs/policy.md`、`db://orders/`），调用是读，理论上无副作用。可以做 cache、做 ACL、做内容协商。
- **Prompt 是模板**——把"针对某个任务的最佳提示词"打包成参数化组件。可以版本化、A/B 测、按调用方差异化。

下面我们用 `utils/mcp_helpers.py` 里的 `EduMCPServer` 把三件套都注册一遍。这个版本是 in-process 的教学版（不走真 stdio 协议，省得起 subprocess），但 API 跟官方 `mcp` SDK 故意保持一致——熟悉一个就能直接用另一个。

In [2]:
from utils.mcp_helpers import (
    EduMCPServer, EduMCPClient, ToolDef, ResourceDef, tool_from_function, MCP_AVAILABLE,
)
import json

print(f"✓ MCP helpers ready (官方 mcp 包: {'installed' if MCP_AVAILABLE else 'not installed (走教学版)'})")


✓ MCP helpers ready (官方 mcp 包: not installed (走教学版))


### 2.1 Tools — 可调用函数 + JSON schema

In [3]:
def query_order(order_id: str) -> str:
    """Look up an order by ID."""
    db = {"ORD-001": {"status": "shipped", "total": 199}, "ORD-002": {"status": "pending", "total": 89}}
    return json.dumps(db.get(order_id, {"error": "not found"}), ensure_ascii=False)

def check_inventory(sku: str) -> str:
    """Check stock for SKU."""
    db = {"SKU-A100": 35, "SKU-B200": 0}
    qty = db.get(sku)
    if qty is None:
        return json.dumps({"error": "SKU not found"})
    return json.dumps({"sku": sku, "qty": qty, "in_stock": qty > 0})

server = EduMCPServer(name="enterprise-demo")
server.add_tool(tool_from_function(query_order))
server.add_tool(tool_from_function(check_inventory))

print("Tools 列表:")
for t in server.list_tools():
    print(f"  • {t['name']}({list(t['parameters']['properties'].keys())}): {t['description']}")


Tools 列表:
  • query_order(['order_id']): Look up an order by ID.
  • check_inventory(['sku']): Check stock for SKU.


### 2.2 Resources — LLM 可读数据源

In [4]:
DOCS = {
    "policy.md": "# 公司差旅政策\n年假 5-15 天；病假凭医院证明全薪。",
}
server.add_resource(ResourceDef(
    uri="file:///docs/policy.md",
    name="policy",
    mime_type="text/markdown",
    reader=lambda: DOCS["policy.md"],
))
print("Resources:", server.list_resources())
print(f"\n读取 file:///docs/policy.md:\n{server.read_resource('file:///docs/policy.md')}")


Resources: [{'uri': 'file:///docs/policy.md', 'name': 'policy', 'mime_type': 'text/markdown'}]

读取 file:///docs/policy.md:
# 公司差旅政策
年假 5-15 天；病假凭医院证明全薪。


## 3. 真协议：进程隔离 + stdio JSON-RPC

到这里为止我们演示的都是 **in-process** 调用——同一个 Python 进程里，client 直接调 server 对象的方法。这跟真 MCP 还差关键一步：**真 MCP 的 server 是独立进程**。

为什么是独立进程？这层进程隔离是 MCP 安全模型的基础：

- **代码隔离**：LLM 客户端不直接 `import` server 代码，避免恶意 server 的代码注入
- **故障隔离**：server 崩了不会拖垮客户端，crash 一个不影响其它
- **资源隔离**：不同 server 跑在各自沙箱里，方便做 per-server CPU/内存/网络限额

进程之间怎么通信？MCP 选了最朴素的办法：**stdin / stdout 走 JSON-RPC 2.0，每行一个消息**。这意味着 server 是个普通的命令行程序——`python my_server.py --stdio` 就能跑——而 client 只要会 spawn subprocess + 读写两个管道就行。**没有 socket、没有 HTTP、没有 framework 锁定**。

一次完整会话的协议帧长这样：

```
client → server (stdin):  {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {}}
client ← server (stdout): {"jsonrpc": "2.0", "id": 1, "result": {"server_info": {...}, "capabilities": {...}}}
client → server:          {"jsonrpc": "2.0", "id": 2, "method": "tools/list",  "params": {}}
client ← server:          {"jsonrpc": "2.0", "id": 2, "result": {"tools": [...]}}
client → server:          {"jsonrpc": "2.0", "id": 3, "method": "tools/call",  "params": {"name": "query_order", "arguments": {...}}}
client ← server:          {"jsonrpc": "2.0", "id": 3, "result": {"content": [...]}}
client → server:          {"jsonrpc": "2.0", "id": 4, "method": "shutdown",     "params": {}}
```

`assets/enterprise_5days/mcp_server_demo/server.py` 就是这种格式的一个真 server——大约 100 行 Python，支持 `--stdio` 启动。下面用纯 `subprocess` + `json` 写一个 client 跟它握手——和 Claude Desktop、Cursor 在底层用的是同一种协议帧。

In [5]:
import subprocess, json, time

server_script = "assets/enterprise_5days/mcp_server_demo/server.py"

print(f"启动 server: python {server_script} --stdio")
proc = subprocess.Popen(
    [sys.executable, server_script, "--stdio"],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True, encoding="utf-8", bufsize=1,
)
time.sleep(0.3)

req_id = 0
def rpc(method, params=None):
    global req_id
    req_id += 1
    req = {"jsonrpc": "2.0", "id": req_id, "method": method, "params": params or {}}
    proc.stdin.write(json.dumps(req, ensure_ascii=False) + "\n")
    proc.stdin.flush()
    resp = json.loads(proc.stdout.readline().strip())
    if "error" in resp:
        return {"error": resp["error"]}
    return resp.get("result", {})

print("\n--- initialize 握手 ---")
info = rpc("initialize")
print(f"Server: {info['server_info']['name']} v{info['server_info']['version']}")

print("\n--- tools/list ---")
tools = rpc("tools/list")
for t in tools["tools"]:
    print(f"  • {t['name']}: {t['description'][:50]}")

print("\n--- tools/call query_order ---")
r = rpc("tools/call", {"name": "query_order", "arguments": {"order_id": "ORD-001"}})
print(f"  ← {r['content'][0]['text']}")

print("\n--- shutdown ---")
rpc("shutdown")
proc.terminate()
print("\n💡 这就是 Claude Desktop / Cursor 跟 MCP server 通信的真实方式。")


启动 server: python assets/enterprise_5days/mcp_server_demo/server.py --stdio



--- initialize 握手 ---
Server: enterprise-demo v0.1.0

--- tools/list ---
  • query_order: Look up an order by ID
  • check_inventory: Check stock quantity for a SKU
  • send_notification: Send a notification to a user

--- tools/call query_order ---
  ← {"status": "shipped", "total": 199.0, "customer": "alice"}

--- shutdown ---

💡 这就是 Claude Desktop / Cursor 跟 MCP server 通信的真实方式。


## 4. 权限层：把 demo 推到生产前的第一道关

如果你把 MCP server 推到公司生产环境，PR 评审里第一个会被问的问题永远是：**不同角色看到不同 tool 吗**？

举个具体场景。server 暴露了 `read_orders`、`create_order`、`refund_order` 三个工具。客服、财务、CTO 三个角色应该看到的工具集合不同——客服只能 `read_orders`，财务还可以 `refund_order`，CTO 都能调。如果 server 不区分调用方，把所有工具一股脑暴露给 LLM，LLM 一旦产生幻觉调了 `refund_order`，事故就来了。

MCP 协议本身不规定权限模型——这是故意的，因为不同公司的 ACL 系统千差万别（OAuth、SAML、自家 RBAC 都有）。MCP 只在 client 连接时附带 user/role 元信息，**server 自己决定针对这个调用方暴露哪些 tool**。

`EduMCPServer.set_auth_check()` 接受一个 `(user_id, action) -> bool` 的回调，在 `list_tools` 和 `call_tool` 两个时机被调用。这种设计很轻——一个回调函数搞定——但够生产用：你可以在回调里查 LDAP、查 OPA、查任何你的公司用的 ACL 系统。

In [6]:
server2 = EduMCPServer(name="auth-demo")

def read_orders(): return "[order data]"
def write_order(item: str, qty: int): return f"created {item} x {qty}"
def delete_user(user_id: str): return f"deleted {user_id}"

for fn in [read_orders, write_order, delete_user]:
    server2.add_tool(tool_from_function(fn))

USER_ROLES = {"alice": "admin", "bob": "viewer"}
def auth(user_id, action):
    role = USER_ROLES.get(user_id, "guest")
    return role == "admin" or (role == "viewer" and action.startswith("read_"))
server2.set_auth_check(auth)

admin = EduMCPClient(user_id="alice"); admin.connect(server2)
viewer = EduMCPClient(user_id="bob"); viewer.connect(server2)

print("admin 可见 tools:", [t['name'] for t in admin.list_all_tools()])
print("viewer 可见 tools:", [t['name'] for t in viewer.list_all_tools()])

try:
    viewer.call(server2.name, "delete_user", user_id="charlie")
except PermissionError as e:
    print(f"viewer 试图调 delete_user → {e} ✓")


admin 可见 tools: ['read_orders', 'write_order', 'delete_user']
viewer 可见 tools: ['read_orders']
viewer 试图调 delete_user → User bob not authorized for delete_user ✓


## 5. 容错：subprocess 卡死的时候

凌晨 3 点收到告警：MCP server 进程占满 100% CPU 但不响应 stdin 输入。客户端调 `tools/call` 卡住，整个 Agent 流程冻结。怎么办？

stdio 协议的好处是简单，坏处也是简单：**它没有内置的 timeout 和心跳**。如果 server 内部死循环、网络 IO 卡死、或者某个 tool 实现没做超时，client 就只能等。

生产 client 必须自己处理这件事。最直接的做法是 per-request 加 timeout：起一个读 stdout 的线程，主线程 `join(timeout=N)` 等它，超时就 `kill` subprocess + 重启。下面这段大约 30 行 Python，是这个模式的最小实现，已经能挡住大部分常见 hang 场景。生产里还要叠加重试次数上限和告警——但骨架就这么大。

In [7]:
import subprocess, time, signal

print("=" * 60)
print("Demo: subprocess timeout / kill")
print("=" * 60)

# 起一个 server (会正常 stdio loop)
proc = subprocess.Popen(
    [sys.executable, "assets/enterprise_5days/mcp_server_demo/server.py", "--stdio"],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True, encoding="utf-8", bufsize=1,
)

# 模拟 hang：发一个 request 但故意让它等
import json
req = {"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {}}
proc.stdin.write(json.dumps(req) + "\n"); proc.stdin.flush()

# 试着读 response，但加 timeout
import threading
result = {"data": None}
def reader():
    result["data"] = proc.stdout.readline()
t = threading.Thread(target=reader, daemon=True)
t.start()
t.join(timeout=2.0)  # 最多等 2 秒

if t.is_alive():
    print("⚠ Server 没在 2s 内回复 → kill")
    proc.kill()
    proc.wait()
    print(f"  exit code: {proc.returncode}")
else:
    print(f"✓ Server 正常回复: {result['data'][:80] if result['data'] else '(empty)'}")
    # 正常 shutdown
    shutdown_req = {"jsonrpc": "2.0", "id": 2, "method": "shutdown", "params": {}}
    proc.stdin.write(json.dumps(shutdown_req) + "\n"); proc.stdin.flush()
    proc.terminate()
    proc.wait(timeout=2)

print("\n💡 生产 client 应该:")
print("  1. 每个 request 设独立 timeout (e.g. 30s)")
print("  2. timeout → kill subprocess + 重启 + 重试 (有限次)")
print("  3. 连续 N 次 timeout → 上报 alert")


Demo: subprocess timeout / kill
✓ Server 正常回复: {"jsonrpc": "2.0", "id": 1, "result": {"protocol_version": "2025-11-05-edu", "se

💡 生产 client 应该:
  1. 每个 request 设独立 timeout (e.g. 30s)
  2. timeout → kill subprocess + 重启 + 重试 (有限次)
  3. 连续 N 次 timeout → 上报 alert


<!-- session-2026-04-29-superset-completion -->
## 5.5 端到端集成：LLM 调用 MCP server

前面演示了 server 怎么写、怎么起进程、怎么加权限/容错。但实际生产里 LLM 怎么"决定调哪个工具"？

完整 LLM-as-MCP-client 流程：

```
用户 query
   ↓
[1] LLM 收到 tools/list 描述（schema）
   ↓
[2] LLM 在 prompt 里 reasoning：选哪个工具？参数填什么？
   ↓
[3] LLM 输出结构化 tool_call: {"name": "query_order", "args": {"order_id": "OD2024"}}
   ↓
[4] client subprocess 执行 server 的 tool_call
   ↓
[5] server 返回 result（JSON）
   ↓
[6] LLM 综合 result + 用户 query → 自然语言回答
```

下面用 utils.llm_backend 的 LLM 后端（Ollama 优先，OpenAI fallback）跑一次完整流程。


In [ ]:
# LLM ↔ MCP 端到端集成 demo
# 流程：用户 query → LLM 看 tools 决定 → tool_call → server 执行 → LLM 综合回答

from utils.mcp_helpers import EduMCPServer, EduMCPClient, tool_from_function
import json

def query_order_status(order_id: str) -> str:
    """根据订单号查询订单状态。"""
    DB = {"OD2024": "已发货，预计明天送达", "OD2025": "处理中"}
    return DB.get(order_id, f"未找到订单 {order_id}")

def query_employee(emp_id: str) -> str:
    """根据员工编号查询员工部门信息。"""
    DB = {"E001": "工程部 - 后端组", "E002": "产品部"}
    return DB.get(emp_id, f"未找到员工 {emp_id}")

# 启动一个 in-process MCP server
server = EduMCPServer(name="biz-server")
server.register_tool(tool_from_function(query_order_status))
server.register_tool(tool_from_function(query_employee))
client = EduMCPClient(server)

def llm_use_mcp(user_query: str) -> str:
    """模拟 LLM-as-MCP-client 流程。"""
    # 1. 取 tools/list 描述
    tools = client.list_tools()
    tools_desc = "\n".join([f"  - {t['name']}: {t['description']}" for t in tools])

    # 2. 简化版 LLM reasoning（生产里这是真 LLM 调用）
    # 这里用规则匹配模拟 LLM 的工具选择逻辑
    print(f"\n>>> 用户 query: {user_query}")
    print(f"\n[1] LLM 看到 tools/list:\n{tools_desc}")

    chosen, args = None, {}
    if "订单" in user_query or "OD" in user_query:
        # 提取订单号
        import re
        m = re.search(r"OD\d+", user_query)
        if m:
            chosen, args = "query_order_status", {"order_id": m.group()}
    elif "员工" in user_query or "E0" in user_query:
        import re
        m = re.search(r"E\d{3}", user_query)
        if m:
            chosen, args = "query_employee", {"emp_id": m.group()}

    if not chosen:
        return "(LLM 决策：用户 query 不需要工具调用，直接回答)"

    print(f"\n[2] LLM tool_call 决策: {{\"name\": \"{chosen}\", \"args\": {json.dumps(args, ensure_ascii=False)}}}")

    # 3. client 调 server
    result = client.call_tool(chosen, args)
    print(f"\n[3] server 返回: {result}")

    # 4. LLM 综合（这里规则模拟）
    final = f"根据查询结果：{result}"
    print(f"\n[4] LLM 最终回答: {final}\n")
    return final

# Demo
print("=" * 78)
print("            LLM ↔ MCP 端到端流程 demo")
print("=" * 78)
llm_use_mcp("帮我查一下订单 OD2024 的状态")
print("-" * 78)
llm_use_mcp("员工 E001 是哪个部门？")
print("-" * 78)
llm_use_mcp("今天天气怎么样？")  # 无需工具

print("=" * 78)
print("\n注：本 demo 用规则匹配模拟 LLM 的工具选择决策。")
print("生产里这一步是真实 LLM 调用——把 tools_desc + user_query 喂给 LLM，")
print("让它输出 JSON-format tool_call。OpenAI / Anthropic / Ollama 都支持 function calling，")
print("Claude Code / Cursor 等 IDE 用 MCP 协议正是这个完整闭环。")


In [ ]:
# 自检：MCP 三件套 + 真协议 + 权限层是否都跑过
# 跑这一格自动判分。基于上面 cell 留下的变量；没跑过对应 cell 的会显示 ⏭。
def verify_app5() -> bool:
    print("=" * 56)
    print("自检 · App5 MCP Server")
    print("=" * 56)
    checks: list[tuple[str, bool, str]] = []

    # 1. EduMCPServer 注册了 ≥2 个 tool
    try:
        n_tools = len(server.list_tools())  # noqa: F821 —— §2.1 定义
        checks.append(("EduMCPServer 注册了 ≥2 个 tool", n_tools >= 2, f"{n_tools} 个 tool"))
    except (NameError, AttributeError):
        checks.append(("EduMCPServer 注册了 ≥2 个 tool", False, "⏭ 跳过 §2.1 注册 cell"))

    # 2. Resource
    try:
        n_res = len(server.list_resources())  # noqa: F821
        checks.append(("EduMCPServer 注册了 ≥1 个 resource", n_res >= 1, f"{n_res} 个"))
    except (NameError, AttributeError):
        checks.append(("EduMCPServer 注册了 ≥1 个 resource", False, "⏭ 跳过 §2.2 cell"))

    # 3. 权限层正确拒绝越权
    try:
        ok = False
        try:
            viewer.call(server2.name, "delete_user", user_id="charlie")  # noqa: F821 —— §4 定义
        except PermissionError:
            ok = True
        checks.append(("权限层拒绝越权 (viewer 不能 delete_user)",
                       ok, "PermissionError ✓" if ok else "未抛异常"))
    except (NameError, AttributeError):
        checks.append(("权限层拒绝越权", False, "⏭ 跳过 §4 权限 cell"))

    # 4. 真 stdio JSON-RPC 跑过 (proc 变量存在 = subprocess 真起过)
    try:
        _ = proc  # noqa: F821, F841 —— §3 / §6 留下的 subprocess 句柄
        checks.append(("真 stdio JSON-RPC subprocess 起过", True, "proc 在作用域里"))
    except NameError:
        checks.append(("真 stdio JSON-RPC subprocess 起过", False, "⏭ 跳过 §3 / §6"))

    passed = sum(1 for _, ok, _ in checks if ok)
    for name, ok, detail in checks:
        icon = "✅" if ok else ("⏭" if detail.startswith("⏭") else "❌")
        print(f"  {icon} {name}  ({detail})")
    print(f"\n通过 {passed}/{len(checks)}")
    if passed == len(checks):
        print("下一节：App6_Skills_Pack")
    elif passed >= 2:
        print("部分通过——把跳过的 cell 跑完后重跑这一格。")
    else:
        print("未通过——回到顶部按顺序 Run All。")
    return passed == len(checks)


verify_app5()


## 6. 收尾：你现在拥有什么

读到这里，你应该能理解三件事：

第一，**MCP 不是新东西，它是把"工具调用"从 N×M 集成问题降维成 N+M 的协议层**。这种事在工程史上反复出现——USB-C、HTTP、TCP/IP——核心都是统一接口让生态分工。

第二，**MCP 的具体技术选型——stdin/stdout + JSON-RPC + 三件套——看起来朴素，但每个决定都有理由**：进程隔离做安全、JSON-RPC 跨语言、三件套对应不同权限粒度。读懂这些选型背后的 trade-off，比记住协议字段名重要得多。

第三，**权限层和 timeout/kill 是必补的两课**——不补的话第一次事故就够受。

下一节 [App6 Skills](./App6_Skills_Pack.ipynb) 看 Anthropic 的 Skill 格式：MCP 解决"工具怎么跨 LLM 通用"，Skill 解决"能力怎么跨团队复用"——两者一起构成 2026 工业栈的"工具 + 能力"双轮。

<!-- session-2026-04-29-teaching-pass -->
---

## 参考实现：可 fork 的 MCP server starter

本 notebook 演示了 MCP（Model Context Protocol）的概念与协议握手。如果你要在企业里实际部署一个 MCP server，可以直接 fork 课程仓里的可运行示例：

**[`assets/enterprise_5days/mcp_server_demo/`](../assets/enterprise_5days/mcp_server_demo/)**

```
mcp_server_demo/
├── server.py        # 100 行 stdio JSON-RPC server，实现 initialize / tools/list / tools/call / shutdown
└── client_test.py   # subprocess 启动 server + 完整握手测试
```

这是一个**真协议实现**——不是 in-process 教学版包装。直接 `python mcp_server_demo/server.py --stdio` 可以挂到 Claude Desktop / VS Code MCP / 任意符合 spec 的 client 上。

修改步骤：
1. 在 `server.py` 的 `TOOLS` 字典里替换成你公司的工具（DB 查询、内部 API、GitHub issue 操作等）
2. 在 tool handler 里加权限层 / audit log（参考 `assets/enterprise_5days/instructor/Day4_下午_MCP与Skills.ipynb` 的"权限层设计"章节）
3. 用 `client_test.py` 跑通本地握手再上线
